In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

#### Hypothesis 1 


Songs with artist collaborations exhibit greater chart longevity than solo tracks, even after controlling for initial popularity and artist characteristics.

Analysis:
Run a multiple linear regression
- input: 
    - collaboration type (dummy variable: 0 = solo, 1 = featured collaboration)
    - number of featured artists
    - peak streams in Week 1 (as a control to isolate longevity from the initial spike)
- output: total weeks on chart

Test whether collaboration > 0, with a significance of 0.05. If collaboration  is significant after controlling for Week 1 streams, it confirms a sustained longevity effect rather than a mere spike.



In [2]:
df = pd.read_csv("../data/processed/processed_chart_tracks_enriched_features.csv")

In [3]:
# prep data for h1
# each row represents a song. each song should only appear once

df['date'] = pd.to_datetime(df['date'])

# find the stream of the song in the first week in top 50
first_appearance = (
    df.sort_values('date')
      .groupby(['uri', 'track_name'], as_index=False)
      .first()                          
)
first_appearance = first_appearance.rename(columns={'streams': 'streams_first_week',
                                                     'date':    'first_chart_date',
                                                     'rank':    'entry_rank_in_top50'})     

# count the weeks the song was in the top 50 chart
total_weeks_on_chart = df.groupby(["uri", "track_name"])["rank"].count().reset_index(name = 'total_weeks_on_chart')

df_h1 = pd.merge(total_weeks_on_chart, first_appearance, on = ["uri", "track_name"])

In [4]:
df_h1.columns

Index(['uri', 'track_name', 'total_weeks_on_chart', 'entry_rank_in_top50',
       'artist_names', 'source', 'peak_rank', 'previous_rank',
       'weeks_on_chart', 'streams_first_week', 'first_chart_date', 'tempo',
       'energy', 'zero_crossing_rate', 'spectral_centroid', 'spectral_rolloff',
       'mfcc_1', 'mfcc_2', 'chroma_mean', 'chroma_std', 'artist_count',
       'collaboration_type', 'monthly_listeners', 'popularity_score'],
      dtype='object')

In [69]:
x = df_h1[['streams_first_week', 'collaboration_type', 'entry_rank_in_top50', 'popularity_score', 'previous_rank']]
X = sm.add_constant(x)
Y = df_h1['total_weeks_on_chart']

model_h1 = sm.OLS(Y, X).fit()
results_h1 = model_h1.summary()
results_h1

# fail to reject, no evidence of collaboration effect on longevity 

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                             
================================================================================
Dep. Variable:     total_weeks_on_chart   R-squared:                       0.077
Model:                              OLS   Adj. R-squared:                  0.074
Method:                   Least Squares   F-statistic:                     26.40
Date:                  Thu, 23 Apr 2026   Prob (F-statistic):           1.10e-25
Time:                          15:09:08   Log-Likelihood:                -6299.6
No. Observations:                  1580   AIC:                         1.261e+04
Df Residuals:                      1574   BIC:                         1.264e+04
Df Model:                             5                                         
Covariance Type:              nonrobust                                         
=======================================================================================
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  13.9061      3.236      4.297      0.000       7.559      20.253
streams_first_week   1.647e-07   3.94e-08      4.179      0.000    8.74e-08    2.42e-07
collaboration_type     -0.7858      0.684     -1.148      0.251      -2.128       0.556
entry_rank_in_top50    -0.1516      0.033     -4.632      0.000      -0.216      -0.087
popularity_score       -0.0719      0.034     -2.144      0.032      -0.138      -0.006
previous_rank           0.0613      0.009      6.499      0.000       0.043       0.080
==============================================================================
Omnibus:                     1421.494   Durbin-Watson:                   2.062
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            53418.864
Skew:                           4.153   Prob(JB):                         0.00
Kurtosis:                      30.248   Cond. No.                     2.37e+08
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.37e+08. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

#### Hypothesis 2: 

Songs released under major labels have greater chart longevity than independent releases, even after controlling for both initial popularity and musical characteristics.

Analysis: 
Run a Multiple Linear Regression
- Output (Dependent Variable): Total weeks on chart
- Input Variables:
    - label_type : dummy variable (0 = independent, 1 = major label), which is the key variable of interest
    - week1_streams :  peak streams in Week, controls for initial popularity
    - danceability, energy, tempo, valence, acousticness :  control for musical characteristics that independently drive longevity

Test: Whether the coefficient for label_type is significantly positive at α = 0.05. If significant after controlling for Week 1 streams and musical features, it confirms a structural label advantage (e.g., marketing budget, playlist placement) rather than just better music or an initial spike.




In [5]:
# H2 Step 1: build df_h2 (row-level chart data + label class)
source_label = pd.read_csv('../data/raw/raw_unique_source_labeled.csv')

# keep only required columns and remove duplicated source rows in mapping table
source_label = source_label[['source', 'major_label']].drop_duplicates(subset=['source'])

df_h2 = df.merge(source_label, on='source', how='left')

print('df_h2 shape:', df_h2.shape)
print('missing major_label rows:', df_h2['major_label'].isna().sum())
print('\nmajor_label distribution (row-level):')
print(df_h2['major_label'].value_counts(dropna=False).sort_index())

df_h2 shape: (13399, 24)
missing major_label rows: 0

major_label distribution (row-level):
major_label
0     832
1    8015
2    4552
Name: count, dtype: int64


In [6]:
# H2 Step 2: convert to song-level dataset (one row per song)
# Outcome = total weeks on chart

df_h2['date'] = pd.to_datetime(df_h2['date'])

first_appearance_h2 = (
    df_h2.sort_values('date')
         .groupby(['uri', 'track_name'], as_index=False)
         .first()
         .rename(columns={
             'streams': 'streams_first_week',
             'date': 'first_chart_date',
             'rank': 'entry_rank_in_top50'
         })
)

total_weeks_h2 = (
    df_h2.groupby(['uri', 'track_name'])['rank']
         .count()
         .reset_index(name='total_weeks_on_chart')
)

df_h2_song = pd.merge(total_weeks_h2, first_appearance_h2, on=['uri', 'track_name'], how='left')

# baseline group = independent (major_label == 2)
df_h2_song['label_big3_parent'] = (df_h2_song['major_label'] == 0).astype(int)
df_h2_song['label_big3_subsidiary'] = (df_h2_song['major_label'] == 1).astype(int)

print('df_h2_song shape:', df_h2_song.shape)
print('\nmajor_label distribution (song-level):')
print(df_h2_song['major_label'].value_counts(dropna=False).sort_index())

df_h2_song shape: (1580, 27)

major_label distribution (song-level):
major_label
0     99
1    880
2    601
Name: count, dtype: int64


In [8]:
# H2 Step 3: run OLS
h2_features = [
    'label_big3_parent', 'label_big3_subsidiary',
    'streams_first_week', 'entry_rank_in_top50', 'popularity_score', 'previous_rank',
    'tempo', 'energy', 'spectral_centroid', 'spectral_rolloff',
    'mfcc_1', 'mfcc_2', 'chroma_mean', 'chroma_std'
]

h2_model_df = df_h2_song.dropna(subset=h2_features + ['total_weeks_on_chart']).copy()

X_h2 = sm.add_constant(h2_model_df[h2_features])
Y_h2 = h2_model_df['total_weeks_on_chart']

model_h2 = sm.OLS(Y_h2, X_h2).fit()
model_h2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                             
================================================================================
Dep. Variable:     total_weeks_on_chart   R-squared:                       0.092
Model:                              OLS   Adj. R-squared:                  0.084
Method:                   Least Squares   F-statistic:                     11.30
Date:                  Sat, 25 Apr 2026   Prob (F-statistic):           3.88e-25
Time:                          17:46:38   Log-Likelihood:                -6287.1
No. Observations:                  1580   AIC:                         1.260e+04
Df Residuals:                      1565   BIC:                         1.268e+04
Df Model:                            14                                         
Covariance Type:              nonrobust                                         
=========================================================================================
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    30.9522     16.539      1.871      0.061      -1.489      63.394
label_big3_parent         0.6205      1.464      0.424      0.672      -2.252       3.493
label_big3_subsidiary     1.6733      0.708      2.362      0.018       0.284       3.063
streams_first_week      1.77e-07   3.97e-08      4.458      0.000    9.91e-08    2.55e-07
entry_rank_in_top50      -0.1473      0.033     -4.502      0.000      -0.211      -0.083
popularity_score         -0.0516      0.034     -1.512      0.131      -0.119       0.015
previous_rank             0.0576      0.010      6.046      0.000       0.039       0.076
tempo                    -0.0057      0.012     -0.459      0.646      -0.030       0.019
energy                  -17.6949      9.138     -1.936      0.053     -35.619       0.229
spectral_centroid        -0.0021      0.004     -0.527      0.598      -0.010       0.006
spectral_rolloff         -0.0013      0.001     -0.853      0.394      -0.004       0.002
mfcc_1                    0.0416      0.012      3.344      0.001       0.017       0.066
mfcc_2                   -0.0385      0.043     -0.899      0.369      -0.123       0.046
chroma_mean              -6.9918      6.073     -1.151      0.250     -18.904       4.920
chroma_std               19.3760     34.758      0.557      0.577     -48.801      87.553
==============================================================================
Omnibus:                     1420.739   Durbin-Watson:                   2.065
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            54566.167
Skew:                           4.139   Prob(JB):                         0.00
Kurtosis:                      30.574   Cond. No.                     2.74e+09
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.74e+09. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [9]:
# H2 Step 4: key coefficient check and conclusion (alpha=0.05)
alpha = 0.05
coef_keys = ['label_big3_parent', 'label_big3_subsidiary']

coef_table = pd.DataFrame({
    'coef': model_h2.params[coef_keys],
    'p_value': model_h2.pvalues[coef_keys]
})
coef_table['significant_positive'] = (coef_table['coef'] > 0) & (coef_table['p_value'] < alpha)

print(coef_table)

if coef_table['significant_positive'].any():
    print('\nConclusion: At least one Big-3 group shows a significant positive longevity effect vs independent labels.')
else:
    print('\nConclusion: No significant positive longevity advantage for Big-3 groups vs independent labels under this model.')

                           coef   p_value  significant_positive
label_big3_parent      0.620463  0.671841                 False
label_big3_subsidiary  1.673260  0.018289                  True

Conclusion: At least one Big-3 group shows a significant positive longevity effect vs independent labels.



#### Hypothesis 3: 

The likelihood of a song becoming a "viral hit" (reaching the Top 15) is significantly higher when specific levels of energy and spectral brightness are combined, rather than the independent effect of either feature alone. 

Analysis: 
Run a Logistic Regression to predict the probability of a song reaching the Top 15

- Input: energy (numerical), spectral_centroid (numerical), and an interaction term (energy × spectral_centroid).
- Output: Viral Status (Dummy variable: 1 if peak_rank is 1–15, 0 if 16–50)

Test: We will test whether the coefficient for the interaction term (interaction) is significantly different from zero ( < 0.05). If significant, it confirms that a certain combo (e.g., high energy plus high brightness) creates a unique synergy that drives virality more effectively than just having one or the other.


In [4]:
# prep data for h3
# each row represents a song, each song should only appear once

df['viral'] = np.where(df['peak_rank'] <= 15, 1, 0)
df_h3 = df.drop_duplicates(subset = ['uri', 'track_name'])

# interaction term 
df_h3['energy_spectral_centroid'] = df_h3['energy'] * df_h3['spectral_centroid']

In [5]:
x = df_h3[['energy', 'spectral_centroid', 'energy_spectral_centroid', 'popularity_score', 'previous_rank']]
Y = df_h3['viral']

X = x.reset_index(drop=True)
Y = Y.reset_index(drop=True)

X = sm.add_constant(X) 
model_h3 = sm.Logit(Y, X).fit()

model_h3.summary()

Optimization terminated successfully.
         Current function value: 0.601737
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                  viral   No. Observations:                 1580
Model:                          Logit   Df Residuals:                     1574
Method:                           MLE   Df Model:                            5
Date:                Thu, 23 Apr 2026   Pseudo R-squ.:                 0.04469
Time:                        15:23:24   Log-Likelihood:                -950.74
converged:                       True   LL-Null:                       -995.22
Covariance Type:            nonrobust   LLR p-value:                 1.118e-17
============================================================================================
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                       -1.2663      0.837     -1.512      0.130      -2.907       0.375
energy                       0.2324      3.301      0.070      0.944      -6.238       6.702
spectral_centroid            0.0002      0.000      0.672      0.502      -0.000       0.001
energy_spectral_centroid    -0.0013      0.002     -0.855      0.393      -0.004       0.002
popularity_score             0.0103      0.006      1.762      0.078      -0.001       0.022
previous_rank               -0.0124      0.002     -6.795      0.000      -0.016      -0.009
============================================================================================
"""